# Rossmann Gesichtspflege Scraper
* Rossman page is a JavaScript-rendered (React/Angular) SPA, BeautifulSoup alone won't work -> Selenium 
* before I scraped all products from all pages, I tested scraping only one product and then only one page
* the the code that scrapes all pages, go to section `Get products from all pages` -> this scrapes over 1100 products and saves them in a csv file
* total runtime: 5-6 hours

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd
from urllib.parse import urlparse
from selenium.common.exceptions import NoSuchElementException

# Configuration and Cookie Banner

In [2]:
options = Options()
driver = webdriver.Chrome(options=options)

url = "https://www.rossmann.de/de/pflege-und-duft/gesichtspflege/c/olcat2_2"
driver.get(url)
driver.implicitly_wait(10)
driver.maximize_window()
wait = WebDriverWait(driver, 15)  # wait up to 15 seconds before element is found

# accept cookie banner
cookie_btn = wait.until(
    EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
)
cookie_btn.click()
# driver.close()

# Get information of one product
Go to the site with all products, click on one product and then extract all relevant information for this product

In [ ]:
card = driver.find_element(By.CSS_SELECTOR, 'div[data-testid="product-card"]')
href = card.find_element(By.CSS_SELECTOR, 'div[data-testid="product-brandAndName"] a').get_attribute("href")
name = card.get_attribute("data-item-name")
brand = card.get_attribute("data-item-brand")
price = card.find_elements(By.CLASS_NAME, 'sr-only')[1].text

driver.get(href)
time.sleep(2)  
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "main")))

def get_details(name):
    try:
        product_details = driver.find_element(By.ID, name)
    except: 
        return ''
    cms_div = product_details.find_element(By.CSS_SELECTOR, "div.rm-cms")
    product_details_html = cms_div.get_attribute("innerHTML")
    # html text to plain text
    product_details_text = BeautifulSoup(product_details_html, "html.parser").get_text(separator="\n")    
    return product_details_text

# Optional: store alongside other fields
doc = {
    "name": name,
    "brand": brand,
    "href": href,
    "product_details": get_details("GRP_PRODUKTDETAILS"),
    "instruction": get_details("GRP_ANWENDUNGGEBRAUCH"),
    "ingredients": get_details("GRP_INHALTSSTOFFE"), 
    "warnings": get_details("GRP_WARNHINWEISE")
}
doc
# driver.quit()


{'name': 'LUMINOUS630® Skin Glow Liquid Refiner',
 'brand': 'NIVEA',
 'href': 'https://www.rossmann.de/de/pflege-und-duft-nivea-luminous630-skin-glow-liquid-refiner/p/4006000171647?rm_tt=eyJhZF9pZCI6ImNmYzM3NzZmLTMxMzEtNGJmZS1iYmU4LTI5ZjVjMWJmNzY4NyIsInByb2plY3RfaWQiOiJkZGY0YzYyMy1iYTBkLTQ2ZjEtOThjMi03OWRhMjVkYmExNjYiLCJza3UiOiI0MDA2MDAwMTcxNjQ3IiwiaWF0IjoxNzc5Mjk4Nzg0LCJsaSI6IjhhYjU4ZjgxLTdhYzAtNGU4YS05NjRkLTdiY2FkMTZhZTRkMyIsImFzIjoiNzczZDQyNTQtZjAxMi00ZmIzLTlkMDEtMmMwN2E4YzU4Yzk3IiwiYmMiOjU3LCJidCI6MSwicGwiOiIzYzU5ZTFjNi1jMzlmLTRjNmQtYmYwYi1iZDMzMzc2OTJhNTUiLCJhdCI6MTc3OTI5ODc4NDU2Nn0%3D.Rmzqgema3SMiR2aiDsAZ5g6xDd4Xc_N2NbHEpRinmHE%3D',
 'product_details': '\n\nDer NIVEA LUMINOUS630® Skin Glow Liquid Refiner ist ein \n1. EBENMÄSSIGES HAUTBILD: sorgt für eine ebenmäßige Haut in 7 Tagen\n2. PORENVERFEINERUNG: verfeinert sofort die Poren und glättet die Haut\n3. SANFTES PEELING: erneuert die Haut durch aktive Inhaltsstoffe 4% AHA & 1% PHA\nDie einzigartige hochwirksame Formel ist leicht

# Get information of all products from one page

In [14]:
cards = driver.find_elements(By.CSS_SELECTOR, 'div[data-testid="product-card"]')
products = []
for card in cards:
    products.append({
    'href': card.find_element(By.CSS_SELECTOR, 'div[data-testid="product-brandAndName"] a').get_attribute("href"),
    'name': card.get_attribute("data-item-name"),
    'brand': card.get_attribute("data-item-brand"),
    'price': card.find_elements(By.CLASS_NAME, 'sr-only')[1].text
    })

def get_details(name):
    """This function returns the description details of a product and returns an empty string if this description does not exist (e.g. sometimes there are no warnings for a product)."""
    try:
        product_details = driver.find_element(By.ID, name)
    except: 
        return ''
    cms_div = product_details.find_element(By.CSS_SELECTOR, "div.rm-cms")
    product_details_html = cms_div.get_attribute("innerHTML")
    # html text to plain text
    product_details_text = BeautifulSoup(product_details_html, "html.parser").get_text(separator="\n")    
    return product_details_text

details = []
for product in products:
    driver.get(product['href'])
    time.sleep(2)  # small buffer for dynamic content
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "main")))


    details.append({
        # "name": name,
        # "brand": brand,
        # "href": href,
        "product_details": get_details("GRP_PRODUKTDETAILS"),
        "instruction": get_details("GRP_ANWENDUNGGEBRAUCH"),
        "ingredients": get_details("GRP_INHALTSSTOFFE"), 
        "warnings": get_details("GRP_WARNHINWEISE")
    })

df_products = pd.DataFrame.from_dict(products)
df_details = pd.DataFrame.from_dict(details)
result = pd.concat([df_products, df_details], axis=1)
result
    # driver.quit()


,href,name,brand,price,product_details,instruction,ingredients,warnings
0,https://www.rossmann.de/de/pflege-und-duft-niv...,CC Fluid Luminous630° Anti-Pigmentflecken LSF ...,NIVEA,"Artikelpreis 26,99 €",\n\nDas LUMINOUS630® ANTI-PIGMENTFLECKEN CC FL...,\n\nTäglich morgens auf das gereinigte Gesicht...,"\n\nAqua, Glycerin, C12-15 Alkyl Benzoate, Met...",\n\nWarnhinweise\nVerwenden Sie die Gesichtspf...
1,https://www.rossmann.de/de/pflege-und-duft-niv...,Cellular Epigenetics Verjüngendes Serum,NIVEA,"Artikelpreis 15,99 €",\n\nDas Cellular Epigenetics Verjüngendes Seru...,"\n\nZwei Mal täglich anwenden, morgens und abe...","\n\nAqua, Glycerin, Cocoglycerides, Methylprop...",
2,https://www.rossmann.de/de/pflege-und-duft-isa...,Augenpads Hydrogel Hibiskus,ISANA,"Artikelpreis 0,79 €",\n\nWirkung: Die ISANA Hydrogel Augenpads mit ...,\n\n1. Folie aufreißen und Augenpads herausneh...,"\n\nAqua, Propylene Gylcol, Glycerin, Butylene...",
3,https://www.rossmann.de/de/pflege-und-duft-isa...,Korean Skincare Hydration Cream,ISANA,"Artikelpreis 5,99 €",\n\nDas Geheimnis strahlender Haut!\nEntdecke ...,\n\nEine großzügige Menge Creme auf das gerein...,"\n\nAQUA, DIPROPYLENE GLYCOL, GLYCERIN, NIACIN...",
4,https://www.rossmann.de/de/pflege-und-duft-isa...,Hydrogel Augenpads,ISANA,"Artikelpreis 0,79 €",\n\nDie ISANA Hydrogel Augenpads mit der Wirks...,\n\n1. Folie aufreißen und Augenpads herausneh...,"\n\nAqua, Glycerin, Sorbitol, Trehalose, Chond...",\n\nWarnhinweise\nMaske zum einmaligen Gebrauc...
5,https://www.rossmann.de/de/pflege-und-duft-isa...,Calming Barrier Serum,ISANA,"Artikelpreis 5,99 €",\n\nWirkstofffkomplex aus:\n5% Ceramid-Komplex...,\n\nMorgens und Abends täglich nach der Reinig...,"\n\nAQUA, GLYCERIN, OCTYLDODECANOL, NIACINAMID...",\n\nWarnhinweise\nKontakt mit Textilien vermeiden
6,https://www.rossmann.de/de/pflege-und-duft-isa...,120h* Liquid Moisturizer,ISANA,"Artikelpreis 3,99 €",\n\nDer zu Allem passende Feuchtigkeits-Kick. ...,\n\nAnwendung: Morgens und abends nach der Rei...,"\n\nAqua, Glycerin, Panthenol, Betaine, Sorbit...",
7,https://www.rossmann.de/de/pflege-und-duft-niv...,Cellular Expert Filler Hochwirksame Anti-Age N...,NIVEA,"Artikelpreis 16,99 €",\n\nNIVEA Cellular Expert Filler - Hochwirksam...,\n\nTäglich abends auf das gründlich gereinigt...,"\n\nAqua, Glycerin, Butyrospermum Parkii Butte...",
8,https://www.rossmann.de/de/make-up-nivea-exper...,Expert Finish Cellular 3in1 Pflege Cushion 02 ...,NIVEA,"Artikelpreis 14,99 €",\n\nNIVEA® Cellular Expert Finish 3in1 Pflege ...,\n\nDrücken Sie das beiliegende Kissen leicht ...,"\n\nAqua, Dimethicone, Glycerin, CI 77891, Eth...",\n\nWarnhinweise\nBitte Augenkontakt vermeiden.
9,https://www.rossmann.de/de/pflege-und-duft-isa...,Augenpads Hydrogel Vitamin C,ISANA,"Artikelpreis 0,79 €",\n\nDie ISANA HYDROGEL AUGENPADS mit der Wirks...,\n\n1. Folie aufreißen und Augenpads herausneh...,"\n\nAqua, Glycerin, Chondrus Crispus Powder, G...",


## Get products from all pages
* runtime: 5-6 h
* problems:
    * when the loop reaches the last page, the next pageIndex does not lead to an error or page without products -> you get redirected to the next section (e.g. shower gel/hair products) which is why my original break condition did not work
    * the Rossmann website seems to have a bug where in some pages the products appear twice because the cards in the background somehow -> since the runtime is this long and I have already wasted time because of the problem with the redirection that I figured out too late, I decided to keep the code this way and clean up the duplicates later in a different script
    * I tried to parallelize everything to shorten the runtime, but it did not work so I ran this over night

In [6]:
options = Options()
driver = webdriver.Chrome(options=options)

base_url = "https://www.rossmann.de/de/pflege-und-duft/gesichtspflege/c/olcat2_2"
expected_path = urlparse(base_url).path

driver.get(base_url)
driver.implicitly_wait(10)
driver.maximize_window()
wait = WebDriverWait(driver, 15)

cookie_btn = wait.until(
    EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
)
cookie_btn.click()

def get_details(name):
    try:
        product_details = driver.find_element(By.ID, name)
        cms_div = product_details.find_element(By.CSS_SELECTOR, "div.rm-cms")
        product_details_html = cms_div.get_attribute("innerHTML")
        return BeautifulSoup(product_details_html, "html.parser").get_text(separator="\n", strip=True)
    except NoSuchElementException:
        return ""

def extract_price(card):
    sr_only_elements = card.find_elements(By.CLASS_NAME, "sr-only")
    texts = [el.text.strip() for el in sr_only_elements if el.text.strip()]
    return texts[1] if len(texts) >= 2 else ""

all_products = []
all_details = []
page_index = 0

while True:
    url = f"{base_url}?pageIndex={page_index}"
    driver.get(url)

    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "main")))
    time.sleep(2)

    current_path = urlparse(driver.current_url).path

    # after the last page, you get automatically redirected to another section (e.g. hair) and we want to stop there
    if current_path != expected_path:
        print(f"Redirect erkannt: {driver.current_url}")
        print(f"Erwartet: {expected_path}, bekommen: {current_path}")
        break

    cards = driver.find_elements(By.CSS_SELECTOR, 'div[data-testid="product-card"]')

    if not cards:
        print(f"No products found on page {page_index}. Stopping.")
        break

    print(f"Scraping page {page_index}, {len(cards)} products found")

    page_products = []
    for card in cards:
        page_products.append({
            "href": card.find_element(
                By.CSS_SELECTOR,
                'div[data-testid="product-brandAndName"] a'
            ).get_attribute("href"),
            "name": card.get_attribute("data-item-name"),
            "brand": card.get_attribute("data-item-brand"),
            "price": extract_price(card)
        })

    for product in page_products:
        driver.get(product["href"])
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "main")))
        time.sleep(2)

        all_details.append({
            "product_details": get_details("GRP_PRODUKTDETAILS"),
            "instruction": get_details("GRP_ANWENDUNGGEBRAUCH"),
            "ingredients": get_details("GRP_INHALTSSTOFFE"),
            "warnings": get_details("GRP_WARNHINWEISE")
        })

    all_products.extend(page_products)
    page_index += 1

driver.quit()

df_products = pd.DataFrame(all_products)
df_details = pd.DataFrame(all_details)
result = pd.concat([df_products, df_details], axis=1)

Scraping page 0, 24 products found
Scraping page 1, 24 products found
Scraping page 2, 24 products found
Scraping page 3, 24 products found
Scraping page 4, 24 products found
Scraping page 5, 24 products found
Scraping page 6, 24 products found
Scraping page 7, 24 products found
Scraping page 8, 24 products found
Scraping page 9, 24 products found
Scraping page 10, 24 products found
Scraping page 11, 24 products found
Scraping page 12, 24 products found
Scraping page 13, 24 products found
Scraping page 14, 24 products found
Scraping page 15, 24 products found
Scraping page 16, 24 products found
Scraping page 17, 24 products found
Scraping page 18, 24 products found
Scraping page 19, 24 products found
Scraping page 20, 24 products found
Scraping page 21, 24 products found
Scraping page 22, 24 products found
Scraping page 23, 24 products found
Scraping page 24, 48 products found
Scraping page 25, 48 products found
Scraping page 26, 48 products found
Scraping page 27, 48 products found
Sc

In [7]:
result['product_details'] = result.product_details.str.strip()
result['instruction'] = result.instruction.str.strip()
result['ingredients'] = result.ingredients.str.strip()
result['warnings'] = result.warnings.str.strip()

In [8]:
result.to_csv('products.csv', index=False)